# Simulation-Based Inference of $\Lambda$CDM from the CMB

We do the measurement of cosmological parameters from the primary CMB (*Planck* temperature auto power spectrum) **without evaluating a likelihood during inference at all**.
The recipe is:

1. run the forward model on parameters drawn from the prior to get pairs $(\boldsymbol\theta, \mathbf{d})$,
2. train a neural network to output a conditional density $q(\boldsymbol\theta|\mathbf{d})$,
3. evaluate that network at the data you actually observed.

This is **neural posterior estimation** (NPE), the most widely used flavour of
*simulation-based inference* (SBI). It is how people do inference when the likelihood is
unknown or intractable.

The CMB is *not* such a problem: the likelihood is Gaussian to a very good approximation. That is exactly why it
is a good place to learn SBI. **We can compute the right answer and grade the network against
it** — a luxury you will not have in a real application, which makes it all the more useful to learn the basics.

### Our model

Essentially ΛCDM but with $\tau_{\rm reio}$ held fixed because with TT alone we cannot break its degeneracy with the scalar fluctuation amplitude $A_s$.


| parameter | meaning |
| --------- | ------- |
| $\omega_b = \Omega_b h^2$ | baryon density |
| $\omega_c = \Omega_c h^2$ | cold dark matter density |
| $h$ | Hubble parameter, $H_0 = 100\,h\,$km/s/Mpc |
| $n_s$ | scalar spectral index |
| $A = \ln(10^{10}A_s e^{-2\tau})$ | primordial amplitude, in the combination TT actually constrains |

### We will

1. build the forward model, and turn the *likelihood* into a *simulator*,
2. compute the right answer with MCMC, so that we have something to compare against,
3. draw 11 000 simulations — 10 000 to train on, 1 000 held out,
4. train normalizing flows (a MAF and an NSF) to give us the posterior,
5. and then, most importantly, work out whether we are allowed to believe the machine learning by running an internal consistency check (simulation-based calibration).

The whole notebook runs in about ten minutes on a CPU, most of it spent training. On
Colab you might be able to speed it up by turning on an accelerator: *Runtime $\to$ Change runtime type $\to$ T4 GPU*.


## 1. Setup

We need three things: the *Planck* plik-lite likelihood (for its bandpower binning, its
covariance matrix and its data), a CosmoPower emulator so the forward model is fast enough to
run tens of thousands of times, and the `sbi` package.

In [ ]:
!test -d planck-lite-py || git clone -q https://github.com/heatherprince/planck-lite-py
!pip install -q sbi

import os, sys, pickle, time, urllib.request
import numpy as np
import matplotlib.pyplot as plt

EMULATOR_FILE = 'cmb_TT_NN.pkl'
if not os.path.exists(EMULATOR_FILE):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/alessiospuriomancini/cosmopower/'
        'main/cosmopower/trained_models/CP_paper/CMB/cmb_TT_NN.pkl', EMULATOR_FILE)

sys.path.insert(0, 'planck-lite-py')
from planck_lite_py import PlanckLitePy
np.random.seed(0)

In [ ]:
import warnings
import torch

# two library warnings that are noise here: torch complaining about a deprecated
# call inside nflows, and the CUDA probe when there is no GPU
warnings.filterwarnings('ignore', message='torch.linalg.solve_triangular has its arguments')
warnings.filterwarnings('ignore', message='CUDA initialization')

from sbi.inference import NPE
from sbi.utils import BoxUniform
from sbi.neural_nets import posterior_nn
from sbi.diagnostics import run_sbc, check_sbc, get_nltp

torch.manual_seed(0)

# our batches are small, so handing torch all the cores costs more in thread
# synchronisation than it gains in arithmetic
torch.set_num_threads(min(4, os.cpu_count()))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('sbi', __import__('sbi').__version__, '| torch', torch.__version__, '| device', device)

## 2. The forward model

A CosmoPower neural-network emulator that maps cosmological parameters
to the TT power spectrum in a few milliseconds. The cell below loads it with plain numpy — the
file is a TensorFlow pickle, but the prediction path is only a handful of matrix
multiplications, so we can sidestep TensorFlow entirely.

Every function here is **batched**. It takes an array of parameter vectors and
returns one spectrum per row. That matters a lot now, because we are about to ask for 11 000
spectra instead of one at a time.

In [ ]:
# --- load the emulator (a few matrix multiplications; details not important here) ------
class _ListShim(list):
    def __setstate__(self, state): pass

class _NoTFUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if name == 'ListWrapper' or 'tracking' in module or 'trackable' in module:
            return _ListShim
        return super().find_class(module, name)

def load_emulator(filename):
    with open(filename, 'rb') as f:
        (W, b, alphas, betas, par_mean, par_std, feat_mean, feat_std,
         n_par, parameters, n_modes, modes, n_hidden, n_layers, arch
         ) = _NoTFUnpickler(f).load()
    return dict(W=[np.asarray(w) for w in W],   b=[np.asarray(x) for x in b],
                alphas=[np.asarray(x) for x in alphas], betas=[np.asarray(x) for x in betas],
                par_mean=np.asarray(par_mean),  par_std=np.asarray(par_std),
                feat_mean=np.asarray(feat_mean), feat_std=np.asarray(feat_std),
                parameters=list(parameters), modes=np.asarray(modes), n_layers=int(n_layers))

def emulate_Cl(emu, params):
    x = np.stack([np.atleast_1d(np.asarray(params[k], float)) for k in emu['parameters']], axis=1)
    h = (x - emu['par_mean']) / emu['par_std']
    for i in range(emu['n_layers'] - 1):
        a = h @ emu['W'][i] + emu['b'][i]
        h = (emu['betas'][i] + (1. - emu['betas'][i])/(1. + np.exp(-emu['alphas'][i]*a))) * a
    return 10**((h @ emu['W'][-1] + emu['b'][-1]) * emu['feat_std'] + emu['feat_mean'])
# --------------------------------------------------------------------------------------

emu     = load_emulator(EMULATOR_FILE)
ell     = emu['modes']      # multipoles, 2 ... 2508
T_CMB   = 2.7255e6          # CMB temperature in micro-Kelvin
TAU_REF = 0.0544            # fixed optical depth to reionization

PARAM_NAMES  = ['omega_b', 'omega_cdm', 'h', 'n_s', 'A']
PARAM_LABELS = [r'$\omega_b$', r'$\omega_c$', r'$h$', r'$n_s$',
                r'$A=\ln(10^{10}A_s e^{-2\tau})$']

# Planck 2018 best-fit values (A built from ln10As=3.044 and TAU_REF)
PLANCK = np.array([0.02237, 0.1200, 0.6736, 0.9649, 3.044 - 2*TAU_REF])

# flat priors, chosen to sit inside the emulator's training box
PRIOR_LO = np.array([0.019, 0.090, 0.60, 0.90, 2.80])
PRIOR_HI = np.array([0.025, 0.150, 0.75, 1.02, 3.10])


def theory_Cl(p):
    '''TT spectrum C_ell in microK^2 at ell = 2 ... 2508.

    Batched: p has shape (5,) or (N, 5), the return value always has shape (N, len(ell)).
    '''
    p = np.atleast_2d(np.asarray(p, float))
    params = {'omega_b': p[:, 0], 'omega_cdm': p[:, 1], 'h': p[:, 2],
              'tau_reio': np.full(len(p), TAU_REF), 'n_s': p[:, 3],
              'ln10^{10}A_s': p[:, 4] + 2*TAU_REF}
    return emulate_Cl(emu, params) * T_CMB**2

def theory_Dl(p):
    '''The same spectrum as D_ell = l(l+1)C_l/2pi, which is what one usually plots.'''
    return ell*(ell+1)/(2*np.pi) * theory_Cl(p)

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(ell, theory_Dl(PLANCK)[0])
ax.set_xlabel(r'multipole $\ell$'); ax.set_ylabel(r'$D_\ell^{TT}\ [\mu K^2]$')
ax.set_title('CMB TT spectrum at the Planck best-fit')
plt.show()

t0 = time.time()
theory_Dl(np.random.uniform(PRIOR_LO, PRIOR_HI, size=(2000, 5)))
print(f'{2000/(time.time()-t0):.0f} spectra per second in batch mode')

The familiar acoustic peaks, and a few thousand spectra per second. Generating a training set
of 11 000 simulations is therefore going to cost us seconds, not hours — which is why the CMB
is a comfortable place to *learn* SBI, and also a warning: in a real application the simulator
is usually the expensive part, and the number of simulations is what limits you.

## 3. From a likelihood to a simulator

Part 1's likelihood was a Gaussian in the 215 *Planck* plik-lite TT bandpowers,

$$-2\ln\mathcal{L}(\mathbf{d}|\boldsymbol\theta) = \left[\mathbf{d} - \mathbf{t}(\boldsymbol\theta)\right]^{\sf T}
\mathsf{C}^{-1}\left[\mathbf{d} - \mathbf{t}(\boldsymbol\theta)\right],$$

where $\mathbf{t}(\boldsymbol\theta)$ is the theory spectrum *binned* into bandpowers and
$\mathsf{C}$ is the released band-power covariance.

Reading that equation right-to-left instead gives us a **simulator**: bin the theory, then add
one draw of correlated Gaussian noise,

$$\mathbf{d} = \mathbf{t}(\boldsymbol\theta) + \mathsf{L}\mathbf{z},
\qquad \mathbf{z}\sim\mathcal{N}(0,\mathsf{I}),\qquad \mathsf{L}\mathsf{L}^{\sf T} = \mathsf{C}.$$

Same physics, same noise model, same 215 numbers — just run forwards. `PlanckLitePy` does the
binning inside its `loglike`, so we pull the ingredients out and write the binning as a matrix
$\mathsf{B}$, one row per bandpower.

In [ ]:
likelihood = PlanckLitePy(data_directory='planck-lite-py/data',
                          year=2018, spectra='TT', use_low_ell_bins=False)

NBIN   = likelihood.nbintt                # 215 bandpowers, 30 <= ell <= 2508
bl     = likelihood.bval[:NBIN]           # bandpower centres
binfac = bl*(bl+1)/(2*np.pi)              # C_l -> D_l at the bin centres

# the weighted average over ell that the likelihood applies internally, as a matrix
B = np.zeros((NBIN, len(ell)))
for i in range(NBIN):
    lo = likelihood.blmin_TT[i] + likelihood.plmin_TT - ell[0]
    hi = likelihood.blmax_TT[i] + likelihood.plmin_TT - ell[0] + 1
    B[i, lo:hi] = likelihood.bin_w_TT[likelihood.blmin_TT[i]:likelihood.blmax_TT[i]+1]

# bandpower covariance. plik-lite stores C_l, we work in D_l, and that conversion is
# diagonal -- so it rescales the covariance without changing the likelihood at all.
COV   = np.linalg.inv(likelihood.fisher) * np.outer(binfac, binfac)
ICOV  = np.linalg.inv(COV)
COV_L = np.linalg.cholesky(COV)

Dl_data = binfac * likelihood.X_data[:NBIN]          # the real Planck bandpowers
Dl_err  = binfac * likelihood.X_sig[:NBIN]

def model_bandpowers(p):
    '''The 215 binned bandpowers in D_l units [muK^2] -- the noise-free simulator output.'''
    return binfac * (theory_Cl(p) @ B.T)

def simulate(p, rng=np.random):
    '''Run the forward model and add one realisation of Planck's noise.'''
    mu = model_bandpowers(p)
    return mu + rng.standard_normal(mu.shape) @ COV_L.T

Now the data. We could feed the network the real *Planck* bandpowers, but for a first pass it
is much more instructive to use a **mock**: then we know the true parameters, and any
disagreement is the network's fault rather than the emulators's.

So: pick $\boldsymbol\theta_\star$ = the *Planck* best fit, run the simulator once, and treat
the result as our observation.

In [ ]:
theta_star = PLANCK.copy()
rng   = np.random.default_rng(42)
x_obs = simulate(theta_star, rng)[0]

fig, ax = plt.subplots(figsize=(7,4))
ax.errorbar(bl, Dl_data, yerr=Dl_err, fmt='.', ms=4, lw=1, color='0.6',
            label='real Planck 2018')
ax.errorbar(bl, x_obs, yerr=Dl_err, fmt='.', ms=4, lw=1, color='C0',
            label=r'our mock $\mathbf{d}_{\rm obs}$')
ax.plot(ell, theory_Dl(theta_star)[0], 'C3', lw=1, label=r'truth $\theta_\star$')
ax.set_xlabel(r'multipole $\ell$'); ax.set_ylabel(r'$D_\ell^{TT}\ [\mu K^2]$')
ax.set_xlim(0, 2000); ax.legend()
ax.set_title('the mock observation we will analyse')
plt.tight_layout(); plt.show()

Indistinguishable from the real thing by eye, as it should be — same signal, same noise
covariance, different noise draw.

## 4. The right answer, by MCMC

We know the likelihood, so let us use it once — not as part of the SBI machinery, but to
produce the **reference posterior** that every plot from here on is measured against.

This section is part 1 in compressed form. The only difference is that `log_like` now compares
to our mock `x_obs` instead of the real bandpowers, and that it uses the covariance directly
rather than going through `PlanckLitePy` (faster, and we just proved the two are identical).

In [ ]:
def log_like(p):
    r = x_obs - model_bandpowers(p)[0]
    return -0.5 * r @ ICOV @ r

def log_prior(p):
    if np.all(p >= PRIOR_LO) and np.all(p <= PRIOR_HI):
        return 0.0
    return -np.inf

def log_posterior(p):
    lp = log_prior(p)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_like(p)

> The next cell implements a single Metropolis-Hastings transition: given the current state,
> its log-posterior, and the Cholesky factor of the proposal covariance, draw a proposal,
> decide whether to accept it, and return the next state.

In [ ]:
def mh_step(x, log_p, prop_chol):
    '''Perform one Metropolis-Hastings transition.

    Parameters
    ----------
    x         : current position in parameter space (array of 5 numbers)
    log_p     : log-posterior already evaluated at x  (so we don't recompute it)
    prop_chol : Cholesky factor L of the proposal covariance;
                a proposal is x' = x + L @ z  with  z ~ N(0, I)

    Returns
    -------
    (x_next, log_p_next, accepted)
    '''

    # 1. propose a new point from the (symmetric) Gaussian proposal
    proposal = x + prop_chol @ np.random.standard_normal(len(x))

    # 2. evaluate the log-posterior there
    log_p_prop = log_posterior(proposal)

    # 3. Metropolis acceptance test (symmetric proposal -> ratio = posterior ratio)
    if np.log(np.random.random()) < log_p_prop - log_p:
        return proposal, log_p_prop, True    # accept: move to the proposal
    else:
        return x, log_p, False               # reject: stay put


In [ ]:
def run_chain(start, prop_chol, n_steps):
    x = np.array(start, dtype=float)
    log_p = log_posterior(x)
    chain   = np.empty((n_steps, len(x)))
    log_ps  = np.empty(n_steps)
    n_accept = 0
    for i in range(n_steps):
        x, log_p, accepted = mh_step(x, log_p, prop_chol)
        chain[i]  = x
        log_ps[i] = log_p
        n_accept += accepted
    print(f'acceptance rate: {n_accept/n_steps:.2f}')
    return chain, log_ps

Two-stages: a short pilot with a diagonal proposal to learn the parameter
covariance, then the real run with a well-scaled proposal.

In [ ]:
np.random.seed(1)

# pilot: rough per-parameter step sizes, no knowledge of the correlations
pilot_steps = np.array([4e-5, 4e-4, 2e-3, 1.2e-3, 4e-3])
pilot_chain, _ = run_chain(np.array([0.0222, 0.119, 0.680, 0.962, 2.940]),
                           np.diag(pilot_steps), 2000)

# proposal covariance from the pilot, with the usual 2.38/sqrt(d) scaling
cov = np.cov(pilot_chain[500:].T)
prop_chol = np.linalg.cholesky(cov) * 2.38 / np.sqrt(5)

# main run
np.random.seed(2)
chain, log_ps = run_chain(np.array([0.0231, 0.115, 0.690, 0.945, 2.915]), prop_chol, 15000)
mcmc_samples = chain[2000:]
print(f'{len(mcmc_samples)} reference samples')

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(8, 8), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(chain[:, i], lw=0.5)
    ax.axhline(theta_star[i], color='C3', lw=1, ls='--')
    ax.set_ylabel(PARAM_LABELS[i])
axes[-1].set_xlabel('step'); axes[0].set_title('reference chain (dashed: truth)')
plt.tight_layout(); plt.show()

Well mixed, and wandering around the truth. Let us also check the width of this posterior
against an independent calculation. Near its peak the posterior is very nearly Gaussian, so
the Fisher matrix

$$F_{ij} = \frac{\partial \mathbf{t}}{\partial\theta_i}^{\sf T}\mathsf{C}^{-1}
\frac{\partial \mathbf{t}}{\partial\theta_j}$$

should predict the $1\sigma$ errors. If the chain and the Fisher matrix agree, neither is
badly wrong — and we can trust the reference.

In [ ]:
eps  = np.array([1e-5, 1e-4, 1e-3, 1e-3, 1e-3])
grad = np.stack([(model_bandpowers(theta_star + eps[i]*np.eye(5)[i])[0] -
                  model_bandpowers(theta_star - eps[i]*np.eye(5)[i])[0]) / (2*eps[i])
                 for i in range(5)])
fisher_sigma = np.sqrt(np.diag(np.linalg.inv(grad @ ICOV @ grad.T)))

print(f'{"parameter":12s} {"MCMC mean":>12s} {"MCMC sigma":>12s} {"Fisher sigma":>14s} {"ratio":>8s}')
for i, nm in enumerate(PARAM_NAMES):
    m, s = mcmc_samples[:, i].mean(), mcmc_samples[:, i].std()
    print(f'{nm:12s} {m:12.5f} {s:12.5f} {fisher_sigma[i]:14.5f} {s/fisher_sigma[i]:8.2f}')

Agreement at the few-percent level. `mcmc_samples` is now our ground truth.

## 5. Simulations instead of a likelihood

Now the SBI part. We draw parameters from the prior, simulate a mock dataset for each, and
hand the resulting pairs to a neural network. Nothing in what follows will ever call
`log_posterior` again.

We generate 11 000 simulations and split them:

* **10 000 for training.** `sbi` will hold back 10% of these internally as its own validation
  set, which it uses to decide when to stop training.
* **1 000 held out**, which the network never sees. These are what we use to compare
  architectures fairly and — crucially — to run the calibration check in §7. (We will only
  need a couple of hundred of them for that; the rest are there for you to play with.)

In [ ]:
prior = BoxUniform(low=torch.tensor(PRIOR_LO, dtype=torch.float32),
                   high=torch.tensor(PRIOR_HI, dtype=torch.float32))

N_SIM = 11_000
rng   = np.random.default_rng(1)

theta_all = rng.uniform(PRIOR_LO, PRIOR_HI, size=(N_SIM, 5))
t0 = time.time()
x_all = simulate(theta_all, rng)
print(f'{N_SIM} simulations in {time.time()-t0:.1f} s')

theta = torch.tensor(theta_all, dtype=torch.float32)
x     = torch.tensor(x_all,     dtype=torch.float32)

theta_train, x_train = theta[:10_000], x[:10_000]
theta_test,  x_test  = theta[10_000:], x[10_000:]
x_obs_t = torch.tensor(x_obs, dtype=torch.float32)

print(f'train {tuple(x_train.shape)}   held out {tuple(x_test.shape)}')

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
for k in range(60):
    ax.plot(bl, x_all[k], color='C0', alpha=0.15, lw=0.7)
ax.plot([], [], color='C0', label='simulations from the prior')
ax.errorbar(bl, x_obs, yerr=Dl_err, fmt='.', ms=4, lw=1, color='C3', label='the observation')
ax.set_xlabel(r'multipole $\ell$'); ax.set_ylabel(r'$D_\ell^{TT}\ [\mu K^2]$')
ax.set_xlim(0, 2000); ax.legend()
ax.set_title('the training set brackets the data')
plt.tight_layout(); plt.show()

The prior is wide: it produces spectra with peaks in quite the wrong places, and the
observation is one unremarkable member of the family. That is the point — the network has to
learn the whole map from spectrum to parameters, not just a local expansion around the answer.

It is also the difficulty. The prior box is between 16 and 56 posterior $\sigma$ wide in each
direction, so the posterior occupies something like $10^{-7}$ of the prior volume, and the
network has to find it from 10 000 examples.

Finally, a plotting helper.

In [ ]:
from scipy.ndimage import gaussian_filter

def corner_plot(sample_sets, labels, names=None, truths=None, colors=None,
                bins=40, smooth=1.0, lims=None):
    '''Corner plot of one or more sample sets. sample_sets is a list of (N, d) arrays.'''
    if isinstance(sample_sets, np.ndarray):
        sample_sets = [sample_sets]
    colors = colors or ['C0', 'C2', 'C4', 'C1']
    d = sample_sets[0].shape[1]
    fig, axes = plt.subplots(d, d, figsize=(2.1*d, 2.1*d))
    if lims is None:
        lims = [(min(s[:, k].min() for s in sample_sets),
                 max(s[:, k].max() for s in sample_sets)) for k in range(d)]
    for i in range(d):
        for j in range(d):
            ax = axes[i, j]
            if j > i:
                ax.axis('off'); continue
            for s, c in zip(sample_sets, colors):
                if i == j:
                    ax.hist(s[:, i], bins=bins, range=lims[i], color=c,
                            histtype='step', density=True)
                    ax.set_yticks([]); ax.set_xlim(*lims[i])
                else:
                    H, xe, ye = np.histogram2d(s[:, j], s[:, i], bins=bins,
                                               range=[lims[j], lims[i]])
                    H = gaussian_filter(H, smooth)
                    flat = np.sort(H.ravel())[::-1]
                    csum = np.cumsum(flat); csum /= csum[-1]
                    t68 = flat[np.searchsorted(csum, 0.68)]
                    t95 = flat[np.searchsorted(csum, 0.95)]
                    xc = 0.5*(xe[:-1]+xe[1:]); yc = 0.5*(ye[:-1]+ye[1:])
                    ax.contour(xc, yc, H.T, levels=[t95, t68], colors=c, linewidths=1.0)
                    ax.set_xlim(*lims[j]); ax.set_ylim(*lims[i])
            if truths is not None:
                ax.axvline(truths[j], color='C3', ls='--', lw=1)
                if i != j:
                    ax.axhline(truths[i], color='C3', ls='--', lw=1)
            if i == d-1:
                ax.set_xlabel(labels[j]); ax.tick_params(axis='x', rotation=45)
            else:
                ax.set_xticklabels([])
            if j == 0 and i != 0:
                ax.set_ylabel(labels[i])
            else:
                ax.set_yticklabels([])
    if names is not None:
        for nm, c in zip(names, colors):
            axes[0, d-1].plot([], [], color=c, label=nm)
        axes[0, d-1].axis('off'); axes[0, d-1].legend(loc='center', frameon=False)
    fig.align_labels()
    plt.tight_layout()
    return fig, lims

## 6. Neural posterior estimation

The model we train is a **normalizing flow**: an invertible map between a simple base
distribution (a unit Gaussian in 5 dimensions) and the posterior we want, whose parameters are
produced by a neural network reading the data vector $\mathbf{d}$. Writing it
$q_\phi(\boldsymbol\theta|\mathbf{d})$, we train it by maximising

$$\sum_{i} \ln q_\phi(\boldsymbol\theta_i|\mathbf{d}_i)$$

over our simulated pairs. In the limit of infinite simulations and infinite flexibility the
maximum of that sum is exactly the true posterior $p(\boldsymbol\theta|\mathbf{d})$ — and not
just for one dataset, but for *every* $\mathbf{d}$ at once. This is the payoff of SBI: one
training run **amortises** over all possible data, and getting the posterior for a new
observation afterwards is a single forward pass.

`sbi` gives us two standard flow families:

* **MAF** — masked autoregressive flow. Each transform is an affine (shift-and-scale) map whose
  coefficients depend autoregressively on the preceding parameters. Cheap and stable, but each
  layer can only do something affine.
* **NSF** — neural spline flow. Same structure, but each transform is a monotonic rational
  spline with `num_bins` knots instead of a straight line: much more expressive per layer, at
  the cost of more parameters.

**When to stop training.** `sbi` holds back 10% of our 10 000 pairs as its own validation set,
watches the loss on it, and stops when it has not improved for `stop_after_epochs` epochs —
then rolls the weights back to the best epoch. That matters here, because these flows *do*
overfit: keep training past the optimum and the held-out performance gets worse.

There is a trap worth knowing about. The roll-back only happens when early stopping actually
fires. If you cut training short with `max_num_epochs` instead, you keep whatever weights the
final epoch happened to have — past the optimum, the overfit ones. So we let early stopping
decide when to quit.

In [ ]:
def train_npe(model='maf', stop_after_epochs=20, theta=None, x=None, **kwargs):
    '''Train a flow on the training set and return (posterior, trainer).'''
    density_estimator = posterior_nn(model=model, **kwargs)

    trainer = NPE(prior=prior, density_estimator=density_estimator,
                  device=device, show_progress_bars=False)
    trainer.append_simulations(theta_train if theta is None else theta,
                               x_train if x is None else x)

    t0 = time.time()
    # deliberately no max_num_epochs: we want early stopping to fire, so that
    # sbi restores the best-validation weights rather than the last ones
    estimator = trainer.train(training_batch_size=200, stop_after_epochs=stop_after_epochs)
    print(f'\nstopped after {trainer.summary["epochs_trained"][-1]} epochs '
          f'({time.time()-t0:.0f} s), best validation loss '
          f'{trainer.summary["best_validation_loss"][-1]:.3f}')
    return trainer.build_posterior(estimator), trainer

We judge a trained network by two numbers and one picture.

* **Held-out $-\ln q(\boldsymbol\theta_\star|\mathbf{d})$**, averaged over pairs the network
  never saw: how much density does it put on the parameters that actually generated each
  dataset? Lower is better. It rewards being narrow *and* correct, and because it averages over
  the whole prior it grades the amortised posterior rather than one lucky dataset.
* **The posterior width at `x_obs`, relative to the MCMC reference.** A ratio near 1 is the
  goal. Larger means the network is throwing information away; smaller means it is
  overconfident, which is much worse.
* And the corner plot against the reference contours.

In [ ]:
N_VALID = 200      # how many held-out pairs to use for the metrics and the checks below

RESULTS = {}       # name -> metrics, filled in as we go

def evaluate(name, posterior, n_samples=20_000, reference=True):
    t0 = time.time()
    nltp = get_nltp(theta_test[:N_VALID], x_test[:N_VALID], posterior).mean().item()
    s = posterior.sample((n_samples,), x=x_obs_t, show_progress_bars=False).cpu().numpy()

    ref_sig, ref_mean = mcmc_samples.std(0), mcmc_samples.mean(0)
    ratio = s.std(0) / ref_sig
    bias  = (s.mean(0) - ref_mean) / ref_sig
    RESULTS[name] = dict(nltp=nltp, ratio=ratio, bias=bias, samples=s)

    print(f'{name}:  held-out -log q(theta*|d) = {nltp:.3f}   '
          f'(over {N_VALID} pairs, {time.time()-t0:.0f} s)\n')
    print(f'{"parameter":12s} {"NPE mean":>11s} {"NPE sigma":>11s} '
          f'{"ref sigma":>11s} {"sigma ratio":>12s} {"bias/ref":>10s}')
    for i, nm in enumerate(PARAM_NAMES):
        print(f'{nm:12s} {s[:, i].mean():11.5f} {s[:, i].std():11.5f} '
              f'{ref_sig[i]:11.5f} {ratio[i]:12.2f} {bias[i]:+10.2f}')

    sets  = [mcmc_samples, s] if reference else [s]
    names = ['MCMC reference', name] if reference else [name]
    corner_plot(sets, PARAM_LABELS, names=names, truths=theta_star,
                colors=['0.5', 'C0'] if reference else ['C0'])
    plt.show()
    return s

In [ ]:
posterior_maf, trainer_maf = train_npe('maf', hidden_features=64, num_transforms=5)
_ = evaluate('maf', posterior_maf)


That looks encouraging. The contours sit essentially on top of the reference, they lean the
right way in every panel — the strong $\omega_c$-$h$ and $\omega_c$-$A$ degeneracies are
reproduced — and the widths come out within about 15% of the chain's. A neural network that
never evaluated a likelihood has recovered the *Planck* posterior from 10 000 forward
simulations.

Hold on to your scepticism, though. Two things in that table should nag at you. The means are
offset from the reference by up to $\sim\!0.6\sigma$, which is more than the chain's own noise.
And some of the widths came out *smaller* than the reference — which, if it is real and not a
fluke of this one dataset, is the bad direction in which to be wrong.


Now the same thing with splines instead of affine transforms.

In [ ]:
posterior_nsf, trainer_nsf = train_npe('nsf', hidden_features=64, num_transforms=5, num_bins=10)
_ = evaluate('nsf', posterior_nsf)


Interesting. The NSF reaches a distinctly better validation loss and a better held-out score,
and its offsets from the reference are smaller. But its contours are visibly *wider* — some
25-40% wider than the chain's, where the MAF's were within 15%.

So which one is better? By eye at `x_obs`, the MAF. By held-out score, the NSF. Those two
verdicts genuinely disagree, and settling it is what §7 is for. Notice meanwhile that the width
comparison is a sample of size one, while the held-out score averages over 200 datasets.


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
for name, tr in [('maf', trainer_maf), ('nsf', trainer_nsf)]:
    ax.plot(tr.summary['validation_loss'], lw=1, label=f'{name} (validation)')
ax.set_xlabel('epoch'); ax.set_ylabel('loss')
ax.set_title('what early stopping was watching')
ax.legend(); plt.tight_layout(); plt.show()

## 7. Can we trust it? Simulation-based calibration

We have posteriors. They came out of a neural network trained on a finite number of
simulations, so there is no reason yet to believe a word of them. Here we happen to have the
MCMC reference to compare against — but that is a luxury of this notebook, not of real
applications, so we need a check that works without one.

That check is **simulation-based calibration**
([Talts et al. 2018](https://arxiv.org/abs/1804.06788)), and the whole idea fits in one
sentence:

> If $\boldsymbol\theta_\star$ is drawn from the prior, $\mathbf{d}$ is simulated from it, and
> we sample from a *correct* posterior $p(\boldsymbol\theta|\mathbf{d})$, then
> $\boldsymbol\theta_\star$ is statistically indistinguishable from those posterior samples.

If the truth is indistinguishable from the samples, then its **rank** among them — how many
samples fall below it — must be uniformly distributed between $0$ and the number of samples.
Any departure from uniformity is a miscalibration, and the *shape* of the departure tells you
which one.

Our held-out simulations are exactly what this needs: parameters drawn from the prior, each
with its own simulated dataset. So let us get posterior samples for $N=200$ of them — starting
with the NSF, and coming back to the MAF once the machinery works. `sample_batched` does all
200 at once, which is much faster than looping.

In [ ]:
N_POST = 1_000            # posterior samples per validation dataset

theta_valid = theta_test[:N_VALID]
x_valid     = x_test[:N_VALID]

t0 = time.time()
# shape (N_POST, N_VALID, 5) -> transpose to (N_VALID, N_POST, 5), one block per dataset
valid_samples = posterior_nsf.sample_batched((N_POST,), x=x_valid,
                                             show_progress_bars=False).permute(1, 0, 2)
print(f'{N_VALID} posteriors x {N_POST} samples in {time.time()-t0:.0f} s')

# hand them over as plain numpy, which is all the exercise needs
theta_valid_np   = theta_valid.numpy()
valid_samples_np = valid_samples.cpu().numpy()
print('true parameters :', theta_valid_np.shape)
print('posterior samples:', valid_samples_np.shape)

> ### ✏️ Exercise
> Everything you need is now in `theta_valid_np` (shape `(200, 5)`) and `valid_samples_np`
> (shape `(200, 1000, 5)`). Implement the calibration check yourself.
>
> 1. **`sbc_ranks`** — for each validation dataset and each parameter, count how many of the
>    1 000 posterior samples fall below the true value. That count is the rank, an integer
>    between 0 and 1 000. No loops needed: broadcasting `theta_true[:, None, :]` against the
>    samples gets you there in one line.
> 2. **`plot_sbc_ranks`** — one histogram per parameter. Overlay what uniformity would look
>    like: with $N$ datasets in $n_{\rm bins}$ bins the expected count per bin is
>    $N/n_{\rm bins}$, and the scatter around it is binomial,
>    $\sigma = \sqrt{N\,p\,(1-p)}$ with $p = 1/n_{\rm bins}$. Shade $\pm 2\sigma$ so it is
>    obvious when a deviation means something.
>
> Then read your own plots. What do the following possible histogram shapes imply:
> - flat
> - $\cup$-shaped, piled up at both ends
> - $\cap$-shaped, piled up in the middle
> - sloped (like a ramp up or down)

In [ ]:
# ===========================================================================
# EXERCISE CELL  --  delete the bodies between the markers and fill them in yourself
# ---------------------------------------------------------------------------
def sbc_ranks(theta_true, post_samples):
    '''Rank of the true parameters among the posterior samples.

    Parameters
    ----------
    theta_true   : (N, d) true parameters, one row per validation dataset
    post_samples : (N, M, d) posterior samples, one block per validation dataset

    Returns
    -------
    ranks : (N, d) integers in 0 ... M
    '''
    # ----------------------------- BEGIN SOLUTION --------------------------
    # broadcast the truth against the sample axis and count how many are below it
    return (post_samples < theta_true[:, None, :]).sum(axis=1)
    # ------------------------------ END SOLUTION ---------------------------


def plot_sbc_ranks(ranks, n_post, labels, n_bins=20):
    '''One rank histogram per parameter, with the uniform expectation shaded.'''
    # ----------------------------- BEGIN SOLUTION --------------------------
    n_valid, d = ranks.shape
    fig, axes = plt.subplots(1, d, figsize=(2.6*d, 2.8), sharey=True)

    # if the ranks were uniform, each bin would hold this many, +/- this much
    p        = 1.0 / n_bins
    expected = n_valid * p
    sigma    = np.sqrt(n_valid * p * (1 - p))

    for i, ax in enumerate(axes):
        ax.axhspan(expected - 2*sigma, expected + 2*sigma, color='0.85',
                   label=r'uniform $\pm 2\sigma$' if i == 0 else None)
        ax.axhline(expected, color='0.4', lw=1, ls='--')
        ax.hist(ranks[:, i], bins=n_bins, range=(0, n_post),
                color='C0', histtype='stepfilled', alpha=0.4)
        ax.hist(ranks[:, i], bins=n_bins, range=(0, n_post),
                color='C0', histtype='step')
        ax.set_xlabel(labels[i]); ax.set_xticks([0, n_post])
    axes[0].set_ylabel('count')
    axes[0].legend(fontsize='small', loc='lower left')
    fig.suptitle('SBC rank histograms', y=1.02)
    plt.tight_layout()
    return fig
    # ------------------------------ END SOLUTION ---------------------------
# ===========================================================================

ranks_nsf = sbc_ranks(theta_valid_np, valid_samples_np)
print('ranks:', ranks_nsf.shape, ' range', ranks_nsf.min(), '...', ranks_nsf.max())
plot_sbc_ranks(ranks_nsf, N_POST, PARAM_LABELS)
plt.show()


With 20 bins you should *expect* roughly one
bin per panel to stray outside $\pm2\sigma$ by chance.


And the same check for the MAF, to see whether the calibration verdict agrees with the ranking
we got from the held-out score.

In [ ]:
valid_samples_maf = posterior_maf.sample_batched((N_POST,), x=x_valid,
                                                 show_progress_bars=False).permute(1, 0, 2)
ranks_maf = sbc_ranks(theta_valid_np, valid_samples_maf.cpu().numpy())
plot_sbc_ranks(ranks_maf, N_POST, PARAM_LABELS)
plt.suptitle('SBC rank histograms -- maf', y=1.02)
plt.show()

ranks_sbi_maf, dap_maf = run_sbc(theta_valid, x_valid, posterior_maf,
                                 num_posterior_samples=N_POST, show_progress_bar=False)
stats_maf = check_sbc(ranks_sbi_maf, theta_valid, dap_maf, num_posterior_samples=N_POST)
RESULTS['maf']['ks_min'] = stats_maf['ks_pvals'].min().item()
RESULTS['nsf']['ks_min'] = stats['ks_pvals'].min().item()


What do you observe? How does the calibration compare between MAF and NSF?